In [6]:
import joblib
import pandas as pd
import numpy as np

print("--- Robust Production Inference and What-If Test ---")

# 1. Modeli diskten yükle
loaded_xgb = joblib.load("../models/xgboost_rain_model.pkl")
feature_names = loaded_xgb.get_booster().feature_names

# 2. X_test hafızada yoksa bile hata almamak için veriyi temiz kaynaktan çekelim
df_temp = pd.read_csv("../data/processed/weather_final.csv")
X_pure = df_temp.drop('RainTomorrow', axis=1)

# 3. Gerçek, tüm şehir ve rüzgar bilgileri eksiksiz olan 1 satırı şablon olarak kopyala
template_row = X_pure.iloc[[0]].copy()

# -------------------------------------------------------------------------
# SCENARIO A: Güneşli, Kuru ve Yüksek Basınçlı Bir Gün Yapalım
# -------------------------------------------------------------------------
scenario_sunny = template_row.copy()
scenario_sunny['Humidity3pm'] = 15.0       # Kupkuru hava
scenario_sunny['Pressure3pm'] = 1030.0     # Çok yüksek anti-siklon basınç
scenario_sunny['Sunshine'] = 12.0          # Full güneş
scenario_sunny['Cloud3pm'] = 0.0           # Sıfır bulut
scenario_sunny['Rainfall'] = 0.0           # Yağış yok
scenario_sunny['MaxTemp'] = 28.0
scenario_sunny['MinTemp'] = 14.0
scenario_sunny['Temp_Range'] = 28.0 - 14.0
scenario_sunny['Humidity_Temp_Interaction'] = 15.0 * 28.0
scenario_sunny = scenario_sunny[feature_names] # Sırala

# -------------------------------------------------------------------------
# SCENARIO B: Kapalı, Yoğun Nemli ve Alçak Basınçlı Bir Gün Yapalım
# -------------------------------------------------------------------------
scenario_stormy = template_row.copy()
scenario_stormy['Humidity3pm'] = 95.0      # Boğucu yüksek nem
scenario_stormy['Pressure3pm'] = 990.0      # Korkunç düşük fırtına basıncı
scenario_stormy['Sunshine'] = 0.0          # Güneş sıfır
scenario_stormy['Cloud3pm'] = 8.0          # Gökyüzü kapkara bulut
scenario_stormy['Rainfall'] = 25.0         # Yoğun yağış var
scenario_stormy['MaxTemp'] = 16.0
scenario_stormy['MinTemp'] = 12.0
scenario_stormy['Temp_Range'] = 16.0 - 12.0
scenario_stormy['Humidity_Temp_Interaction'] = 95.0 * 16.0
scenario_stormy = scenario_stormy[feature_names] # Sırala

# 4. Tahminleri Çalıştır
prob_sunny = loaded_xgb.predict_proba(scenario_sunny)[0][1]
pred_sunny = loaded_xgb.predict(scenario_sunny)[0]

prob_stormy = loaded_xgb.predict_proba(scenario_stormy)[0][1]
pred_stormy = loaded_xgb.predict(scenario_stormy)[0]

# --- GERÇEKÇİ SONUÇLARI YAZDIR ---
print("\n" + "="*50)
print("CORRECTED RESULTS FOR WHAT-IF SCENARIOS")
print("="*50)
print(f"☀️ SUNNY DAY SCENARIO:")
print(f" -> Predicted Rain Probability: {prob_sunny*100:.2f}%")
print(f" -> Final Binary Decision (Rain Tomorrow?): {'YES (1)' if pred_sunny == 1 else 'NO (0)'}")

print("\n⛈️ STORMY DAY SCENARIO:")
print(f" -> Predicted Rain Probability: {prob_stormy*100:.2f}%")
print(f" -> Final Binary Decision (Rain Tomorrow?): {'YES (1)' if pred_stormy == 1 else 'NO (0)'}")
print("="*50)

--- Robust Production Inference and What-If Test ---

CORRECTED RESULTS FOR WHAT-IF SCENARIOS
☀️ SUNNY DAY SCENARIO:
 -> Predicted Rain Probability: 31.48%
 -> Final Binary Decision (Rain Tomorrow?): NO (0)

⛈️ STORMY DAY SCENARIO:
 -> Predicted Rain Probability: 93.42%
 -> Final Binary Decision (Rain Tomorrow?): YES (1)
